# Explaining models with SHAP: worked solution

Worked solution for the exercise in the lesson **[Explaining models with SHAP](https://ramkumarmd.com/medical-ai/machine-learning/explainability-shap)** (*Machine learning* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](07-explainability-shap.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** SHAP (SHapley Additive exPlanations) breaks a model's prediction for one patient into contributions from each input, which add up to the prediction. Averaged across patients, it shows which inputs the model relies on most, and it can reveal interactions that simpler importance scores miss. SHAP describes how the model behaves, not why the patient is at risk, and its explanations become unreliable when inputs are strongly correlated.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q shap

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Gradient-boosted trees for readmission risk](https://ramkumarmd.com/medical-ai/machine-learning/gradient-boosting-readmission)**: Synthetic discharges with non-linear effects and an interaction built in.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(8)
n = 12000
df = pd.DataFrame({
    "age": rng.integers(20, 95, n),
    "length_of_stay": rng.gamma(2.0, 2.5, n).round(1),
    "prior_admissions": rng.poisson(0.8, n),
    "sodium": rng.normal(138, 4, n).round(),
    "heart_failure": rng.binomial(1, 0.15, n),
    "lives_alone": rng.binomial(1, 0.25, n),
})
logit = (-2.6 + 0.35 * np.minimum(df.prior_admissions, 4)
         + 0.04 * np.maximum(df.age - 65, 0)
         + 0.9 * (np.abs(df.sodium - 139) > 6)        # risk at both low and high sodium
         + 0.6 * df.heart_failure
         + 0.8 * df.heart_failure * df.lives_alone    # interaction
         + 0.05 * df.length_of_stay)
df["readmit_30d"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
print(f"{n} synthetic discharges, 30-day readmission {df.readmit_30d.mean():.1%}")

# 12000 synthetic discharges, 30-day readmission 18.8%

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Global explanation: what the model relies on

Run the data-generation block from the gradient-boosting lesson first. Install with: pip install shap

In [ ]:
import shap
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

features = ["age", "length_of_stay", "prior_admissions", "sodium",
            "heart_failure", "lives_alone"]
train, test = train_test_split(df, test_size=0.3, random_state=0, stratify=df.readmit_30d)
model = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=500, max_leaf_nodes=15,
                                       early_stopping=True, random_state=0)
model.fit(train[features], train.readmit_30d)

explainer = shap.TreeExplainer(model)
sv = explainer(test[features])            # SHAP values on the log-odds scale
global_importance = pd.Series(np.abs(sv.values).mean(axis=0), index=features)
print(global_importance.sort_values(ascending=False).round(3))

# age                 0.270
# heart_failure       0.164
# prior_admissions    0.151
# length_of_stay      0.124
# sodium              0.110
# lives_alone         0.027

### Local explanation: one patient

Continuing from the previous block: explain the prediction for one synthetic patient with heart failure who lives alone.

In [ ]:
i = test.reset_index(drop=True).query("heart_failure == 1 and lives_alone == 1").index[0]
patient = sv[i]
print(test[features].iloc[i].to_string())
print(f"\nBaseline log-odds {patient.base_values:.2f}")
for name, value in sorted(zip(features, patient.values), key=lambda t: -abs(t[1])):
    print(f"{name:<18} {value:+.2f}")
print(f"Model log-odds    {patient.base_values + patient.values.sum():.2f}")

# age                  78.0
# length_of_stay        7.8
# prior_admissions      1.0
# sodium              145.0
# heart_failure         1.0
# lives_alone           1.0
#
# Baseline log-odds -1.55
# heart_failure      +0.65
# length_of_stay     +0.36
# lives_alone        +0.25
# age                +0.20
# sodium             +0.05
# prior_admissions   +0.00
# Model log-odds    -0.03

### Finding an interaction

Continuing from the previous block: the contribution of living alone, split by heart-failure status.

In [ ]:
t = test.reset_index(drop=True)
alone = pd.Series(sv[:, "lives_alone"].values)
print("Mean SHAP value of lives_alone = 1:")
print(alone[t.lives_alone == 1].groupby(t.heart_failure[t.lives_alone == 1]).mean().round(2).to_string())

# Mean SHAP value of lives_alone = 1:
# heart_failure
# 0    0.03
# 1    0.19

### When features are correlated

Continuing from the 'Global explanation' block: add a near-duplicate of age and re-explain.

In [ ]:
rng2 = np.random.default_rng(1)
train2 = train.assign(age_registry=train.age + rng2.normal(0, 1, len(train)))
test2 = test.assign(age_registry=test.age + rng2.normal(0, 1, len(test)))
features2 = features + ["age_registry"]
model2 = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=500, max_leaf_nodes=15,
                                        early_stopping=True, random_state=0)
model2.fit(train2[features2], train2.readmit_30d)
sv2 = shap.TreeExplainer(model2)(test2[features2])
imp2 = pd.Series(np.abs(sv2.values).mean(axis=0), index=features2)
print(imp2[["age", "age_registry"]].round(3).to_string())

# age             0.105
# age_registry    0.193

## Exercise

**Goal:** Use SHAP to explain a readmission model, find its interaction, and test how explanations change with correlated inputs.

**Data:** Synthetic discharges generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the global ranking and draw a beeswarm plot.

In [ ]:
print(global_importance.sort_values(ascending=False).round(3).to_string())
shap.plots.beeswarm(sv)

Each dot is one patient. Its position shows how much that feature moved the patient's predicted log-odds,
and its colour shows the feature's value. For age, high values (red) push risk up; for sodium, both very
low and very high values push it up, which a single coefficient could not show.

### Step 2

Draw waterfall plots for three patients with very different predicted risks.

In [ ]:
p_test = model.predict_proba(test[features])[:, 1]
order = np.argsort(p_test)
picks = {"lowest risk": order[0], "median risk": order[len(order) // 2], "highest risk": order[-1]}
for label, i in picks.items():
    print(f"{label}: predicted risk {p_test[i]:.1%}")
    shap.plots.waterfall(sv[i], show=True)

The waterfall starts at the average log-odds and adds each feature's contribution until it reaches this
patient's prediction. Explanations are on the log-odds scale; large-looking contributions can still mean a
small change in probability for a low-risk patient.

### Step 3

Use SHAP interaction values (explainer.shap_interaction_values) to confirm the heart-failure-by-living-alone interaction.

In [ ]:
interactions = explainer.shap_interaction_values(test[features])
mean_abs = pd.DataFrame(np.abs(interactions).mean(axis=0), index=features, columns=features)
off_diagonal = mean_abs.where(~np.eye(len(features), dtype=bool)).stack().sort_values(ascending=False)
print("Strongest pairwise interactions (mean |SHAP interaction value|, each pair counted twice):")
print(off_diagonal.iloc[::2].head(5).round(4).to_string())

heart_failure × lives_alone is among the strongest pairs (second here), matching the one interaction built
into the simulation. The others (length of stay × age, prior admissions × age...) are not in the simulation:
the trees approximate smooth effects with splits, and some of that shows up as small spurious interactions.
Interaction values point you to candidates; checking them (as in the worked example's grouped means)
is still needed.

### Step 4

Add a leaky feature (for example a 'follow-up booked after readmission' flag) and show how SHAP exposes it.

In [ ]:
rng3 = np.random.default_rng(2)
def with_leak(d):
    # A flag recorded AFTER the outcome: follow-up booked because the patient was readmitted
    booked = np.where(d.readmit_30d == 1, rng3.binomial(1, 0.7, len(d)), rng3.binomial(1, 0.05, len(d)))
    return d.assign(followup_booked_after=booked)

leak_features = features + ["followup_booked_after"]
train_l, test_l = with_leak(train), with_leak(test)
leaky_model = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=500, max_leaf_nodes=15,
                                             early_stopping=True, random_state=0)
leaky_model.fit(train_l[leak_features], train_l.readmit_30d)
from sklearn.metrics import roc_auc_score
print(f"AUROC without leak {roc_auc_score(test.readmit_30d, p_test):.3f}, "
      f"with leak {roc_auc_score(test_l.readmit_30d, leaky_model.predict_proba(test_l[leak_features])[:, 1]):.3f}")
sv_leak = shap.TreeExplainer(leaky_model)(test_l[leak_features])
print(pd.Series(np.abs(sv_leak.values).mean(axis=0), index=leak_features).sort_values(ascending=False).round(3).to_string())

A sudden jump in AUROC together with one feature dominating every explanation is the classic sign of a
leak. SHAP does not know the feature is illegitimate, but it makes the model's dependence on it obvious,
so a clinician reviewing the explanation can ask "when is this recorded?"

### Step 5

Compare SHAP importances with the logistic-regression coefficients from the gradient-boosting lesson and explain the differences.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(train[features])
lr = LogisticRegression(max_iter=2000).fit(scaler.transform(train[features]), train.readmit_30d)
lr_sv = shap.LinearExplainer(lr, scaler.transform(train[features]))(scaler.transform(test[features]))
comparison = pd.DataFrame({
    "LR coefficient (per SD)": lr.coef_[0],
    "LR mean |SHAP|": np.abs(lr_sv.values).mean(axis=0),
    "Boosting mean |SHAP|": global_importance,
}, index=features)
print(comparison.round(3).sort_values("Boosting mean |SHAP|", ascending=False).to_string())

The two models disagree mainly where the truth is non-linear or interacting:

- **Sodium**: risk rises at both low and high values. A linear model can only fit one slope, so its
  coefficient is near zero and sodium looks unimportant; boosting captures the U-shape.
- **Age**: risk rises only after 65. The linear model spreads this over all ages.
- **Lives alone**: matters only with heart failure. Logistic regression without an interaction term gives
  it one average effect for everyone; in boosting its contribution is concentrated in patients with heart
  failure, so its average over all patients is smaller even though it matters a lot for that subgroup.

Neither set of numbers says what would happen if you changed a feature; both describe what each model
relies on.

---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/explainability-shap](https://ramkumarmd.com/medical-ai/machine-learning/explainability-shap). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.